In [ ]:
!pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    duckduckgo-search \
    ddgs \
    langchain-classic

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('APIKEY')
print("Klucz API załadowany pomyślnie.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.tools import Tool

gamma3FakeNews = ChatGoogleGenerativeAI(
    model="gemma-3-27b-it",
    temperature=0.1,
)

react_template = """Jesteś ekspertem w swojej dziedzinie.
{system_prompt}

Masz dostęp do następujących narzędzi:
{tools}

Zawsze używaj poniższego formatu do rozwiązywania problemów:
Question: zadanie od użytkownika
Thought: myślę, co powinienem zrobić krok po kroku
Action: nazwa narzędzia (wybierz tylko z: {tool_names})
Action Input: hasło lub instrukcja dla narzędzia
Observation: wynik działania narzędzia
... (możesz powtarzać sekwencję Thought/Action/Action Input/Observation tyle razy, ile potrzebujesz)
Thought: Znam już ostateczną odpowiedź na podstawie zebranych danych.
Final Answer: ostateczna odpowiedź po polsku.

Question: {input}
Thought:{agent_scratchpad}"""

coordinator_template = """Jesteś Głównym Koordynatorem Systemu AI ds. Weryfikacji Informacji.
Twoim zadaniem jest analiza zapytań użytkownika i przekierowywanie ich do odpowiednich specjalistów.
NIGDY nie szukaj informacji w internecie samodzielnie – zlecaj to swoim podległym agentom!

Historia poprzedniej konwersacji (jeśli istnieje):
{history}

Masz dostęp do następujących agentów (narzędzi):
{tools}

Używaj klasycznego formatu:
Question: zadanie użytkownika
Thought: Muszę zdecydować, do kogo to wysłać.
Action: nazwa agenta (wybierz z: {tool_names})
Action Input: dokładne pytanie do przekazania agentowi
Observation: odpowiedź od podległego agenta
... (powtarzaj w razie potrzeby)
Thought: Otrzymałem raport od agentów, mogę sformułować odpowiedź końcową.
Final Answer:
**Temat analizy:** [Krótko w 1-2 zdaniach określ, czego dotyczyła badana informacja lub tekst]
**Werdykt i Podsumowanie:** [Twoja ostateczna odpowiedź dla użytkownika na podstawie raportów]

Zaczynamy!
Question: {input}
Thought:{agent_scratchpad}"""

coordinator_prompt = PromptTemplate.from_template(coordinator_template)
base_react_prompt = PromptTemplate.from_template(react_template)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_transformer = SentenceTransformer('paraphrase-MiniLM-L6-v2')

baza_wiedzy_lista = [
    "Teoria spiskowa o ptakach-dronach jest fałszywa. Ruch 'Birds Aren't Real' to w rzeczywistości satyryczny projekt artystyczny.",
    "Informacja o tym, że picie wybielacza leczy wirusy, to bardzo niebezpieczny fake news. Wybielacz jest silnie toksyczny dla ludzkiego organizmu.",
    "Zdjęcie Papieża w białej, puchowej kurtce zostało w całości wygenerowane przez sztuczną inteligencję Midjourney.",
    "Sieć 5G nie służy do kontroli umysłów ani nie roznosi wirusów, jest to standard telekomunikacyjny oparty na falach radiowych.",
    "Lądowanie człowieka na Księżycu w 1969 roku miało miejsce naprawdę. Rzekome 'dowody' na nagranie w studiu, jak falująca flaga, zostały wyjaśnione prawami fizyki w próżni.",
    "Szczepionki nie zawierają mikroczipów śledzących. Są to przebadane preparaty medyczne stymulujące układ odpornościowy organizmu.",
    "Ziemia nie jest płaska; dowodzą tego liczne zdjęcia satelitarne, prawa grawitacji oraz setki lat obserwacji astronomicznych."
]

chunks = [fakt.strip() for fakt in baza_wiedzy_lista if len(fakt.strip()) > 5]

embedding_chunks = model_transformer.encode(chunks)

def cosine_similarity(a, b):
    return np.dot(a, b.T) / (np.linalg.norm(a, axis=1)[:, np.newaxis] * np.linalg.norm(b, axis=1))

def zapytaj_wlasny_rag_numpy(pytanie: str) -> str:
    query_vec = model_transformer.encode([pytanie])

    similarity_scores = cosine_similarity(query_vec, embedding_chunks)


    top_indices = np.argsort(similarity_scores[0])[::-1][:2]

    najlepsze_wyniki = []
    for index in top_indices:
        if similarity_scores[0][index] > 0.2:
            najlepsze_wyniki.append(chunks[index])

    if not najlepsze_wyniki:
        return "Brak informacji na ten temat we własnej Bazie Wiedzy. Przeszukaj internet za pomocą innych dostępnych narzędzi (np. Agent_FakeNews)."

    return "Oto pewne i zweryfikowane fakty z Wewnętrznej Bazy RAG:\n" + "\n".join(najlepsze_wyniki)

In [ ]:
net_search = DuckDuckGoSearchRun()
tools = [net_search]

In [ ]:
fakeNewsTemplate = base_react_prompt.partial(
    system_prompt="""Jesteś Głównym Weryfikatorem Faktów (Fact-Checker).
    Twoim zadaniem jest rzetelne i bezstronne sprawdzanie, czy informacje podane przez użytkownika są prawdziwe.

    Zasady Twojego działania:
    1. Zawsze poszukuj dowodów w sieci w celu weryfikacji.
    2. Jeśli znajdziesz dowody potwierdzające – uznaj informację za PRAWDĘ.
    3. Jeśli znajdziesz dowody zaprzeczające – uznaj informację za FAŁSZ.
    4. Jeśli po dokładnym przeszukaniu nie znajdziesz żadnych rzetelnych informacji potwierdzających ani zaprzeczających, wyraźnie stwierdź, że informacja jest NIEWERYFIKOWALNA.
    5. Nigdy nie zakładaj z góry, że sam brak informacji w wynikach wyszukiwania oznacza, że coś jest fałszem."""
)
fakeNewsA = create_react_agent(gamma3FakeNews, tools, fakeNewsTemplate)
fakeNewsExe = AgentExecutor(
    agent=fakeNewsA,
    tools=tools,
    verbose=True
)

In [ ]:
fakeNewsSourceTemplate = base_react_prompt.partial(
    system_prompt="""Jesteś Analitykiem ds. Wiarygodności Źródeł (OSINT).
    Twoim celem jest badanie pochodzenia informacji i ocena rzetelności jej nadawcy.

    Podczas weryfikacji zwróć szczególną uwagę na:
    - Kto jest autorem (czy to anonimowe konto, znany dziennikarz, farma trolli, czy satyryk)?
    - Jaką reputację w sieci ma dany portal lub domena internetowa?
    - Czy strona ma historię publikowania fake newsów, czy jest zaufanym portalem informacyjnym?
    - Czy źródło ma widoczne powiązania polityczne lub ideologiczne, które mogą wpływać na obiektywizm?

    Sformułuj jasny werdykt na temat zaufania do tego źródła."""
)
fakeNewsSourceA = create_react_agent(gamma3FakeNews, tools, fakeNewsSourceTemplate)
fakeNewsSourceExe = AgentExecutor(
    agent=fakeNewsSourceA,
    tools=tools,
    verbose=True
)


In [ ]:
propagandaTemplate = base_react_prompt.partial(
    system_prompt="""Jesteś ekspertem ds. socjotechniki, propagandy i analizy dyskursu.
    NIE szukaj informacji w internecie – polegaj wyłącznie na swoich zdolnościach analitycznych.

    Twoim zadaniem jest ocena samego TEKSTU pod kątem technik manipulacji. Szukaj w szczególności:
    - Języka nacechowanego silnymi emocjami (strach, gniew, sensacja, oburzenie).
    - Zastosowania clickbaitów i wyolbrzymień.
    - Występowania błędów logicznych (np. ataków personalnych, dowodów anegdotycznych).
    - Sformułowań mających na celu polaryzację odbiorców (podział "My" vs "Oni").

    Wypunktuj szczegółowo, krok po kroku, jakich sztuczek lingwistycznych i psychologicznych użył autor, by wpłynąć na podświadomość czytelnika."""
)
propagandaA = create_react_agent(gamma3FakeNews, [], propagandaTemplate) # Pamiętaj o pustej liście narzędzi []
propagandaExe = AgentExecutor(
    agent=propagandaA,
    tools=[],
    verbose=True
)

In [ ]:
import re
from langchain_core.messages import HumanMessage

def zapytaj_fakeNews(pytanie: str) -> str:
    return fakeNewsExe.invoke({"input": pytanie})["output"]

def zapytaj_fakeNewsCorrect(pytanie: str) -> str:
    return fakeNewsCorrectExe.invoke({"input": pytanie})["output"]

def zapytaj_fakeNewsSource(pytanie: str) -> str:
    return fakeNewsSourceExe.invoke({"input": pytanie})["output"]

def zapytaj_propaganda(pytanie: str) -> str:
    return propagandaExe.invoke({"input": pytanie})["output"]

def zapytaj_imageScanner(pytanie: str) -> str:
    znalezione_linki = re.findall(r'(https?://[^\s]+)', pytanie)

    if not znalezione_linki:
        return "Błąd: Nie znalazłem żadnego linku do obrazu. Poproś użytkownika o podanie poprawnego adresu URL."

    url_obrazu = znalezione_linki[0]

    wiadomosc = HumanMessage(
        content=[
            {"type": "text", "text": "Jesteś ekspertem ds. analizy obrazu. Przeanalizuj to zdjęcie i wykonaj polecenie: " + pytanie},
            {"type": "image_url", "image_url": url_obrazu}
        ]
    )

    odpowiedz = gamma3FakeNews.invoke([wiadomosc])
    return odpowiedz.content

In [ ]:
FakeNewsTools = [
    Tool(
        name="Agent_FakeNews",
        func=zapytaj_fakeNews,
        description="Agent służący do sprawdzania, czy informacje są prawdziwe. Korzystaj z niego w pierwszej kolejności, aby zweryfikować fakty."
    ),
    Tool(
        name="Agent_FakeNewsCorrect",
        func=zapytaj_fakeNewsCorrect,
        description="Agent służący do sprawdzenia poprawnej wersji wydarzeń. Używaj go, by zdobyć prawdziwy kontekst, gdy dowiesz się, że informacja to fake news."
    ),
    Tool(
        name="Agent_FakeNewsSource",
        func=zapytaj_fakeNewsSource,
        description="Agent służący do sprawdzania źródła i jego rzetelności. Korzystaj z niego, aby ustalić, skąd pochodzi dany fake news."
    ),
    Tool(
        name="Agent_ImageScanner",
        func=zapytaj_imageScanner,
        description="Używaj tego narzędzia TYLKO wtedy, gdy użytkownik poda w pytaniu link (URL) do zdjęcia. Narzędzie to potrafi przejrzeć obraz pod tym linkiem i go opisać."
    ),
    Tool(
        name="Agent_AnalizaManipulacji",
        func=zapytaj_propaganda,
        description="Używaj, aby przeanalizować podany tekst pod kątem technik manipulacji, emocjonalnego języka, clickbaitów i propagandy. Świetnie uzupełnia weryfikację faktów."
    ),
    Tool(
        name="Agent_BazaWiedzyRAG",
        func=zapytaj_wlasny_rag_numpy,
        description="Zawsze używaj tego narzędzia jako PIERWSZEGO! Służy do przeszukiwania zaufanej, lokalnej bazy faktów. Jeśli to narzędzie zna odpowiedź, nie musisz już szukać w internecie."
    ),
]

In [ ]:
FakeNewsCoordinatorA = create_react_agent(gamma3FakeNews, FakeNewsTools, coordinator_prompt)
FakeNewsCoordinatorExe = AgentExecutor(
    agent=FakeNewsCoordinatorA,
    tools=FakeNewsTools,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
FakeNewsChat = RunnableWithMessageHistory(
    FakeNewsCoordinatorExe,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    output_messages_key="output",
)

In [ ]:
import gradio as gr

def odpowiedz_dla_aplikacji(pytanie):
    print(f"Odebrano zapytanie: {pytanie}")
    try:
        odpowiedz = FakeNewsChat.invoke(
            {"input": pytanie},
            config={"configurable": {"session_id": "sesja_wpf_osint"}}
        )
        return odpowiedz['output']
    except Exception as e:
        return f"Błąd po stronie Agenta: {str(e)}"

#INTERFEJS
with gr.Blocks() as demo:
    with gr.Row():
        input_text = gr.Textbox(label="Pytanie")
        output_text = gr.Textbox(label="Odpowiedź")

    #ENDPOINT API
    btn = gr.Button("Wyślij")
    btn.click(fn=odpowiedz_dla_aplikacji, inputs=input_text, outputs=output_text, api_name="predict")

demo.launch(share=True)